# Causal Inference: Does Loan Modification Actually Reduce Adverse Outcomes?

Most projects that touch causal ML use a synthetic scenario because real treatment/outcome data is hard to find. This dataset has real modification outcomes -- `modification_flag` and each loan's terminal disposition -- letting us ask a genuine causal question instead of a hypothetical one: **among loans that became seriously delinquent, does receiving a modification reduce the probability of ultimately ending in an adverse (foreclosure-type) outcome?**

- **Population**: loans that were ever seriously delinquent (`default_target=1`), restricted to those that have since *resolved* (reached some terminal zero-balance event) -- excludes still-active/unresolved loans, since their eventual outcome is unknown.
- **Treatment**: `ever_modified` -- did the loan receive a modification at any point.
- **Outcome**: `adverse_outcome` -- did the loan terminate via an adverse code (foreclosure, short sale, REO, etc.) vs. a clean resolution (cure, payoff).
- **Honest scope limitation, stated upfront**: available confounders are origination-time features only (credit score, DTI, LTV, etc.). A fully rigorous analysis would also control for delinquency severity/timing at the point modification was offered -- that data isn't readily available in this extraction, so this adjusts for *how risky the loan looked at origination*, not *how distressed it was when modified*. Treat the estimate as a real, methodologically sound first pass, not a definitive causal claim.

## Load data and the naive (unadjusted) comparison

In [1]:
import polars as pl
import pandas as pd
import numpy as np

df = pl.read_parquet("../data/docs/causal_modification_dataset.parquet")
resolved = df.filter(pl.col("adverse_outcome").is_not_null())
print(f"Total defaulted loans: {df.height:,}")
print(f"Resolved (terminal outcome known): {resolved.height:,}")

naive = resolved.group_by("ever_modified").agg(
    n=pl.len(), adverse_rate=pl.col("adverse_outcome").mean()
)
print("\n=== Naive (unadjusted) comparison ===")
print(naive)
print("\n--> Modified loans show a slightly HIGHER raw adverse-outcome rate than unmodified loans.")
print("    This is the classic signature of confounding-by-indication: loans get modified precisely")
print("    because they're already in worse shape, so a naive comparison understates (or reverses) any")
print("    true protective effect. This is exactly the question causal adjustment is for.")

Total defaulted loans: 243,304
Resolved (terminal outcome known): 75,744

=== Naive (unadjusted) comparison ===
shape: (2, 3)
┌───────────────┬───────┬──────────────┐
│ ever_modified ┆ n     ┆ adverse_rate │
│ ---           ┆ ---   ┆ ---          │
│ bool          ┆ u32   ┆ f64          │
╞═══════════════╪═══════╪══════════════╡
│ true          ┆ 5274  ┆ 0.515738     │
│ false         ┆ 70470 ┆ 0.488946     │
└───────────────┴───────┴──────────────┘

--> Modified loans show a slightly HIGHER raw adverse-outcome rate than unmodified loans.
    This is the classic signature of confounding-by-indication: loans get modified precisely
    because they're already in worse shape, so a naive comparison understates (or reverses) any
    true protective effect. This is exactly the question causal adjustment is for.


## DoWhy causal model: identify, estimate, refute

In [2]:
import networkx as nx
# Compatibility shim: DoWhy 0.8 (the latest on PyPI) calls the now-renamed
# networkx.algorithms.d_separated -- newer networkx (3.3+) renamed it to is_d_separator.
# Same signature (G, x, y, z), so this alias is a safe, standard fix rather than downgrading
# networkx (which the segment-correlation notebook also depends on).
if not hasattr(nx.algorithms, "d_separated"):
    nx.algorithms.d_separated = nx.is_d_separator
    nx.d_separated = nx.is_d_separator

from dowhy import CausalModel

CONFOUNDERS = ["credit_score", "original_dti", "original_upb", "original_cltv",
               "original_ltv", "original_interest_rate"]

full_model_df = resolved.select(
    CONFOUNDERS + ["ever_modified", "adverse_outcome"]
).drop_nulls().to_pandas()
full_model_df["ever_modified"] = full_model_df["ever_modified"].astype(int)
full_model_df["adverse_outcome"] = full_model_df["adverse_outcome"].astype(int)
print(f"Full complete-case sample: {len(full_model_df):,}")

# DoWhy's propensity_score_matching does pairwise nearest-neighbor matching, which does not
# scale to 75K+ rows in reasonable time (a first attempt at full scale timed out after 30 min).
# Subsample to a size where matching and all three refutation tests (each re-runs estimation)
# complete quickly, while remaining a large, statistically solid sample.
SAMPLE_SIZE = 15_000
model_df = full_model_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Modeling sample (subsampled for tractable matching): {len(model_df):,}")

causal_model = CausalModel(
    data=model_df,
    treatment="ever_modified",
    outcome="adverse_outcome",
    common_causes=CONFOUNDERS,
)
identified_estimand = causal_model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

Full complete-case sample: 75,553
Modeling sample (subsampled for tractable matching): 15,000
Estimand type: nonparametric-ate

### Estimand : 1
Estimand name: backdoor
Estimand expression:
       d                                                                       ↪
────────────────(E[adverse_outcome|original_ltv,credit_score,original_dti,orig ↪
d[ever_modified]                                                               ↪

↪                                                
↪ inal_cltv,original_upb,original_interest_rate])
↪                                                
Estimand assumption 1, Unconfoundedness: If U→{ever_modified} and U→adverse_outcome then P(adverse_outcome|ever_modified,original_ltv,credit_score,original_dti,original_cltv,original_upb,original_interest_rate,U) = P(adverse_outcome|ever_modified,original_ltv,credit_score,original_dti,original_cltv,original_upb,original_interest_rate)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 

In [3]:
estimate = causal_model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
)
print(f"Estimated ATE (propensity score matching): {estimate.value:+.4f}")
print(f"Interpretation: modification changes the probability of an adverse outcome by {estimate.value:+.2%}")
print(f"(negative = modification reduces adverse-outcome risk once origination-risk confounders are matched)")

propensity_score_matching


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Estimated ATE (propensity score matching): +0.0404
Interpretation: modification changes the probability of an adverse outcome by +4.04%
(negative = modification reduces adverse-outcome risk once origination-risk confounders are matched)


## Naive difference vs. adjusted estimate, side by side

In [4]:
naive_diff = model_df.loc[model_df["ever_modified"] == 1, "adverse_outcome"].mean() - \
             model_df.loc[model_df["ever_modified"] == 0, "adverse_outcome"].mean()

print(f"Naive (unadjusted) difference:      {naive_diff:+.4f}")
print(f"Propensity-matched (adjusted) ATE:  {estimate.value:+.4f}")
print(f"Shift from adjustment:              {estimate.value - naive_diff:+.4f}")

Naive (unadjusted) difference:      +0.0351
Propensity-matched (adjusted) ATE:  +0.0404
Shift from adjustment:              +0.0053


## Refutation tests -- does the estimate survive scrutiny?

DoWhy's refutation suite doesn't prove causality, but it checks whether the estimate is fragile: a random common cause shouldn't change it much, a placebo (fake) treatment should produce an effect near zero, and removing a random subset of the data shouldn't flip the sign.

In [5]:
refute_random = causal_model.refute_estimate(
    identified_estimand, estimate, method_name="random_common_cause"
)
print(refute_random)

refute_placebo = causal_model.refute_estimate(
    identified_estimand, estimate, method_name="placebo_treatment_refuter", placebo_type="permute"
)
print(refute_placebo)

refute_subset = causal_model.refute_estimate(
    identified_estimand, estimate, method_name="data_subset_refuter", subset_fraction=0.8
)
print(refute_subset)

C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Refute: Use a subset of data
Estimated effect:0.0404
New effect:0.04093750000000001
p value:0.98



## Summary

- **Naive comparison**: modified loans show a higher raw adverse-outcome rate (51.6% vs. 48.9%) -- the expected confounding-by-indication signature (worse loans get modified more).
- **Adjusted estimate** (propensity score matching on origination-time confounders): **+0.0404** -- modification remains associated with a *higher* probability of an adverse outcome even after adjustment; the sign did not flip.
- **All three refutation tests pass**, confirming the estimation procedure itself is sound: stable to a random added confounder (0.0404 -> 0.0364, p=0.90), collapses to ~zero under a placebo/fake treatment (0.0404 -> 0.0004), and stable under an 80% data subset (0.0404 -> 0.0409).
- **The honest interpretation**: this does not mean modification causes worse outcomes. The refutation tests validate that the estimation is behaving correctly *given the assumed confounder set* -- they cannot detect omitted-variable bias from a confounder that isn't included. Available confounders here are origination-time only (credit score, DTI, LTV) -- they capture how risky a loan looked *at origination*, not how distressed it was *at the moment modification was offered*, which is very likely the real driver of who gets modified. That missing, more direct confounder (delinquency severity/duration at time of modification) is the natural next refinement, e.g. via a marginal structural model that handles time-varying confounding properly.
- **What this notebook actually demonstrates**: a complete, methodologically real causal inference pipeline (identify -> estimate -> refute) on genuine treatment/outcome data, with an honestly reported result and a clearly identified limitation -- not a synthetic scenario engineered to produce a clean answer.